# JaksuHealth — Kaggle Runner

This notebook clones or updates the public GitHub repository, locates the
attached AMD-SD Kaggle input, validates the dataset and canonical class
mapping, prepares an eye-level grouped split, and runs the selected model
experiment.

**Primary dataset reference:** Hu, Y., Gao, Y., Gao, W. *et al.*
*AMD-SD: An Optical Coherence Tomography Image Dataset for wet AMD Lesions
Segmentation.* Scientific Data 11, 1014 (2024).
https://www.nature.com/articles/s41597-024-03844-6#Sec6

Before running:
1. Attach AMD-SD with **Add Input**.
2. Enable a GPU accelerator.
3. Enable Internet access.
4. Replace `YOUR_GITHUB_USERNAME` below.

In [ ]:
# User settings
GITHUB_USERNAME = "YOUR_GITHUB_USERNAME"
REPOSITORY = "jaksuhealth-ai-model"
BRANCH = "main"

# Choose one: "single" or "all"
RUN_MODE = "single"

# Used only when RUN_MODE == "single"
MODEL_NAME = "unet"  # unet, unetplusplus, or deeplabv3plus

# Set True for a one-epoch pipeline check before a full experiment.
SMOKE_TEST = False

# Dataset split settings
SPLIT_SEED = 42

## 1. Clone or update the GitHub repository

Rerunning this cell updates tracked source files to the selected branch but
does not remove ignored Kaggle outputs such as `data/` and `results/`.

In [ ]:
import os
import subprocess
from pathlib import Path

if GITHUB_USERNAME == "YOUR_GITHUB_USERNAME":
    raise ValueError("Set GITHUB_USERNAME before continuing.")

REPO_DIR = Path("/kaggle/working") / REPOSITORY
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPOSITORY}.git"

if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        [
            "git", "-C", str(REPO_DIR), "reset", "--hard",
            f"origin/{BRANCH}",
        ],
        check=True,
    )
elif REPO_DIR.exists():
    raise RuntimeError(
        f"{REPO_DIR} exists but is not a Git repository. "
        "Rename or remove it before rerunning."
    )
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print("Repository:", Path.cwd())

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print("Dependencies installed.")

## 2. Verify the Kaggle GPU environment

In [ ]:
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. Open Kaggle Notebook settings and select a "
        "GPU accelerator before training."
    )

print("GPU:", torch.cuda.get_device_name(0))

## 3. Locate and validate AMD-SD

The Kaggle mount path is discovered automatically. The validation checks
that image and mask filenames match and that every mask contains only class
IDs 0–5.

In [ ]:
from pathlib import Path

candidates = sorted({
    path.parent
    for path in Path("/kaggle/input").rglob("images")
    if path.is_dir() and (path.parent / "masks").is_dir()
})

if not candidates:
    raise FileNotFoundError(
        "Could not find a Kaggle input containing images/ and masks/. "
        "Attach AMD-SD using Add Input."
    )

if len(candidates) > 1:
    print("Multiple dataset candidates found:")
    for candidate in candidates:
        print(" -", candidate)

DATA_DIR = candidates[0]
IMAGE_DIR = DATA_DIR / "images"
MASK_DIR = DATA_DIR / "masks"

print("AMD-SD directory:", DATA_DIR)

In [ ]:
import cv2
import numpy as np

SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

image_paths = sorted(
    path for path in IMAGE_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)
mask_paths = sorted(
    path for path in MASK_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

image_names = {path.name for path in image_paths}
mask_names = {path.name for path in mask_paths}

missing_masks = sorted(image_names - mask_names)
missing_images = sorted(mask_names - image_names)

if not image_paths:
    raise RuntimeError(f"No supported images found in {IMAGE_DIR}")
if missing_masks or missing_images:
    raise RuntimeError(
        f"Image-mask mismatch. Missing masks: {missing_masks[:5]}; "
        f"missing images: {missing_images[:5]}"
    )

detected_ids: set[int] = set()
for mask_path in mask_paths:
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise RuntimeError(f"Cannot read mask: {mask_path}")
    detected_ids.update(int(value) for value in np.unique(mask))

allowed_ids = {0, 1, 2, 3, 4, 5}
unexpected_ids = detected_ids - allowed_ids
if unexpected_ids:
    raise ValueError(f"Unexpected mask class IDs: {sorted(unexpected_ids)}")

print("Images:", len(image_paths))
print("Masks:", len(mask_paths))
print("Detected class IDs:", sorted(detected_ids))

## 4. Verify the canonical JaksuHealth class mapping

The processed masks used by this project use `2 = IRF` and `3 = PED`.
This mapping must remain identical across training, evaluation,
visualization, and software integration.

In [ ]:
sys.path.insert(0, str(REPO_DIR / "src"))

from jaksuhealth_ai.constants import CLASS_NAMES_BY_ID

EXPECTED_MAPPING = {
    0: "Background",
    1: "SRF",
    2: "IRF",
    3: "PED",
    4: "SHRM",
    5: "IS/OS",
}

if CLASS_NAMES_BY_ID != EXPECTED_MAPPING:
    raise AssertionError(
        f"Class mapping mismatch. Expected {EXPECTED_MAPPING}, "
        f"got {CLASS_NAMES_BY_ID}."
    )

print("Canonical mapping verified:")
for class_id, class_name in CLASS_NAMES_BY_ID.items():
    print(f"  {class_id}: {class_name}")

## 5. Prepare the deterministic eye-level grouped split

The filename prefix before `_` is treated as an eye ID. `--overwrite`
makes this cell safe to rerun with the same seed.

In [ ]:
SPLIT_DIR = REPO_DIR / "data" / "AMD-SD_Split"

subprocess.run(
    [
        sys.executable,
        "scripts/prepare_dataset.py",
        "--data-dir", str(DATA_DIR),
        "--output-dir", str(SPLIT_DIR),
        "--seed", str(SPLIT_SEED),
        "--overwrite",
    ],
    check=True,
)

print((SPLIT_DIR / "split_summary.csv").read_text(encoding="utf-8"))

## 6. Select the experiment

This notebook executes **either** one architecture **or** all architectures,
so a model is not accidentally trained twice.

When `SMOKE_TEST = True`, a temporary one-epoch configuration is generated
to check the pipeline before committing GPU time to the full experiment.

In [ ]:
import copy
from pathlib import Path

import yaml


MODEL_CONFIGS = {
    "unet": Path("configs/unet.yaml"),
    "unetplusplus": Path("configs/unetplusplus.yaml"),
    "deeplabv3plus": Path("configs/deeplabv3plus.yaml"),
}


if RUN_MODE not in {"single", "all"}:
    raise ValueError("RUN_MODE must be either 'single' or 'all'.")


if RUN_MODE == "single":
    if "MODEL_NAME" not in globals():
        raise NameError(
            "MODEL_NAME must be defined when RUN_MODE='single'."
        )

    if MODEL_NAME not in MODEL_CONFIGS:
        raise ValueError(
            f"MODEL_NAME must be one of {sorted(MODEL_CONFIGS)}."
        )

    selected_config = MODEL_CONFIGS[MODEL_NAME]
    selected_run_name = MODEL_NAME

    if SMOKE_TEST:
        config = yaml.safe_load(
            selected_config.read_text(encoding="utf-8")
        )
        config = copy.deepcopy(config)

        selected_run_name = f"{MODEL_NAME}_smoke"

        config["project"]["experiment_name"] = selected_run_name
        config["training"]["output_dir"] = (
            f"results/runs/{selected_run_name}"
        )
        config["training"]["epochs"] = 1
        config["training"]["patience"] = 1
        config["training"]["batch_size"] = min(
            2,
            int(config["training"]["batch_size"]),
        )

        selected_config = (
            Path("/kaggle/working")
            / f"{selected_run_name}.yaml"
        )

        selected_config.write_text(
            yaml.safe_dump(config, sort_keys=False),
            encoding="utf-8",
        )

    print("Run mode:", RUN_MODE)
    print("Model:", MODEL_NAME)
    print("Config:", selected_config)
    print("Run name:", selected_run_name)


else:
    selected_config = None
    selected_run_name = None

    print("Run mode:", RUN_MODE)
    print("Models to run:")

    for model_name, config_path in MODEL_CONFIGS.items():
        print(f"- {model_name}: {config_path}")

## 7. Train, evaluate, and compare

In [ ]:
if RUN_MODE == "single":
    subprocess.run(
        [
            sys.executable,
            "scripts/train.py",
            "--config", str(selected_config),
            "--device", "cuda",
        ],
        check=True,
    )

    checkpoint = Path("results/runs") / selected_run_name / "best_model.pth"
    subprocess.run(
        [
            sys.executable,
            "scripts/evaluate.py",
            "--config", str(selected_config),
            "--checkpoint", str(checkpoint),
            "--split", "test",
            "--device", "cuda",
        ],
        check=True,
    )
else:
    subprocess.run(
        [
            sys.executable,
            "scripts/run_all_experiments.py",
            "--device", "cuda",
        ],
        check=True,
    )

## 8. Review generated outputs

Use **Save Version** in Kaggle after training so the `results/` directory is
preserved as Notebook Output.

In [ ]:
result_files = [
    path for path in sorted(Path("results").rglob("*"))
    if path.is_file()
]

print(f"Generated {len(result_files)} result files:")
for path in result_files:
    print(path)